# 1. Import Libraries, Configuration Setup, and Load the Dataset

## 1.1 Install \& Import Libraries

In [ ]:
# !pip install peft==0.13.2, transformers==4.41.1

In [ ]:
# !pip install wordcloud

In [ ]:
# !pip install imblearn

In [ ]:
# !pip install lightgbm

In [ ]:
# ── Standard library ─────────────────────────────────────
import os
import re
import gc
import json
import time
import random
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, Tuple, Optional

# ── Core DS stack ────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# ── Scikit-learn ─────────────────────────────────────────
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    f1_score, roc_curve, auc, accuracy_score,
)
from sklearn.utils import resample, shuffle

# ── PyTorch ──────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import OneCycleLR

# ── Transformers ─────────────────────────────────────────
from transformers import AutoConfig, AutoTokenizer, AlbertForSequenceClassification, AutoModelForSequenceClassification

# ── Notebook setup ───────────────────────────────────────
warnings.filterwarnings("ignore")


## 1.2 Seed \& Device

In [ ]:
# One notebook copy per training seed. Change RUN_SEED only.
RUN_SEED = 2025

# Fixed R0 data constants. Independent of the training seed, so every seed
# sees the identical split.
SPLIT_SHUFFLE_SEED = 2025

SEED = RUN_SEED

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Run seed: {RUN_SEED} | split shuffle: {SPLIT_SHUFFLE_SEED} | Device: {DEVICE}")


## 1.3 Configuration

In [ ]:
REPO_ROOT = Path(os.environ.get("MENTALHEALTH_REPO_ROOT", Path.cwd())).expanduser().resolve()
DATA_ROOT = Path(os.environ.get("MENTALHEALTH_DATA_ROOT", REPO_ROOT / "0. Dataset")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("MENTALHEALTH_OUTPUT_ROOT", REPO_ROOT / "outputs")).expanduser().resolve()
RAW_DATA_FILE = "mental_health_unified_labels_final.csv"
OUTPUT_DIR = "05_SoftLabel_TrainAll"

ALBERT_MODEL_NAME = "albert-base-v2"
BIOBERT_MODEL_NAME = "dmis-lab/biobert-v1.1"

TEXT_COLUMN = "statement"

CLASS_NAMES = ["SUICIDAL", "DEPRESSION", "STRESS", "ANXIETY", "NORMAL", "BIPOLAR", "PERSONALITY_DISORDER"]
NUM_CLASSES = len(CLASS_NAMES)

SOFT_COLS = ["u_p_suicidal", "u_p_depression", "u_p_stress", "u_p_anxiety", "u_p_normal", "u_p_bipolar", "u_p_personality_disorder"]
LABEL_COLUMN = "u_label"
SPECIAL_LABELS = {"OUT_OF_SCOPE", "INSUFFICIENT"}
LABEL_MAP = {name: i for i, name in enumerate(CLASS_NAMES)}

MAX_TOKEN_LENGTH = 200
BATCH_SIZE = 128
RANDOM_STATE = 42     # split constant, same as R0
PATIENCE = 3          # same as R0 hard
NUM_WORKERS = 2   # same as the R0 hard transformer loaders

# Optimizer settings match the hard-label arm; only the target representation
# and loss differ between the modeling branches.
USE_FP16 = torch.cuda.is_available()
USE_COMPILE = hasattr(torch, "compile") and torch.cuda.is_available()  # R0 hard used compile
USE_ONECYCLE = True
ONECYCLE_PCT_START = 0.1
MAX_GRAD_NORM = 1.0

# Soft cross-entropy is unweighted because per-class weights would rescale
# the distributed score-vector targets.

# R0 hard-style transformer random search. Same search budget and ranges as R0 hard.
N_ITERATIONS = 10
TRANSFORMER_PARAM_SPACE = {
    "lr": (np.log10(1e-5), np.log10(3e-5)),
    "epochs": (4, 10),
    "dropout": (0.0, 0.25),
}

RETRAIN = True

RAW_DATA_PATH = DATA_ROOT / "analysis_ready" / RAW_DATA_FILE
OUTPUT_PATH = OUTPUT_ROOT / OUTPUT_DIR
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

MODEL_PATHS = {
    "albert": {
        "model":  OUTPUT_PATH / "best_albert_soft_all.pth",
        "params": OUTPUT_PATH / "best_albert_soft_all_params.json",
    },
    "biobert": {
        "model":  OUTPUT_PATH / "best_biobert_soft_all.pth",
        "params": OUTPUT_PATH / "best_biobert_soft_all_params.json",
    },
}

print("--- R0-path soft-label TrainAll rerun with R0 hard-style random search ---")
print(f"  Seed      : {RUN_SEED}")
print(f"  Output    : {OUTPUT_PATH}")
print(f"  FP16={USE_FP16}  OneCycle={USE_ONECYCLE}  clip={MAX_GRAD_NORM}")
print("  Class wts : unweighted soft cross-entropy")
print(f"  Random search: {N_ITERATIONS} draws | {TRANSFORMER_PARAM_SPACE}")


## 1.4 Mount drive & Load the data

In [ ]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {RAW_DATA_PATH}. Download the OSF data package "
        "into 0. Dataset/ or set MENTALHEALTH_DATA_ROOT."
    )

df_raw = pd.read_csv(RAW_DATA_PATH, index_col=0)
print(f"Loaded {df_raw.shape[0]} rows.")

df_raw.head()


In [ ]:
# ── SCOPE GATE: keep only evaluable posts ────────────────
before = len(df_raw)
df_raw = df_raw[~df_raw[LABEL_COLUMN].isin(SPECIAL_LABELS)].reset_index(drop=True)
print(f"Scope gating: {before} → {len(df_raw)} rows "
      f"(removed {before - len(df_raw)} OUT_OF_SCOPE/INSUFFICIENT)")
df_raw.head()

In [ ]:
# The data shuffle is a fixed R0 data constant. It must not follow RUN_SEED,
# or each seed would produce a different split.
df_raw = shuffle(df_raw, random_state=SPLIT_SHUFFLE_SEED).reset_index(drop=True)


# 2. Data Exploration \& Master Data Preparation

## 2.1 Master Data Preparation

### 2.1.1 Text cleaning

In [ ]:
def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+", " urltoken ", text)
    text = re.sub(r"@\w+", " usertoken ", text)
    text = re.sub(r"#(\w+)", r" hashtag_\1 ", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


print("Cleaning text...")
df_clean = df_raw.copy()
df_clean[TEXT_COLUMN] = df_clean[TEXT_COLUMN].apply(clean_text)

### 2.1.2 Validate soft-label probability columns

In [ ]:
print("\nValidating soft-label probability columns...")
for col in SOFT_COLS:
    if col not in df_clean.columns:
        raise ValueError(f"Missing column: {col}")
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").fillna(0.0)

# Validate rows sum to ~1
row_sums = df_clean[SOFT_COLS].sum(axis=1)
print(f"  Probability row sums — min: {row_sums.min():.4f}, max: {row_sums.max():.4f}, mean: {row_sums.mean():.4f}")

# Normalize to ensure sum = 1
soft_array = df_clean[SOFT_COLS].to_numpy(dtype=np.float32)
soft_array = soft_array / np.clip(soft_array.sum(axis=1, keepdims=True), 1e-8, None)

for i, col in enumerate(SOFT_COLS):
    df_clean[col] = soft_array[:, i]

# Hard label from u_label column
df_clean["y_hard"] = df_clean[LABEL_COLUMN].map(LABEL_MAP)
n_unmapped = df_clean["y_hard"].isna().sum()
if n_unmapped > 0:
    print(f"  WARNING: {n_unmapped} unmapped labels — dropping")
    df_clean = df_clean.dropna(subset=["y_hard"]).reset_index(drop=True)
    soft_array = df_clean[SOFT_COLS].to_numpy(dtype=np.float32)
df_clean["y_hard"] = df_clean["y_hard"].astype(int)

print(f"\n{LABEL_COLUMN} distribution:")
print(df_clean[LABEL_COLUMN].value_counts())

## 2.2 Master Data Split

In [ ]:
groups = df_clean[TEXT_COLUMN]
y_for_split = df_clean["y_hard"]

# 80/20 → train+val / test
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
trainval_idx, test_idx = next(gss_test.split(df_clean, y_for_split, groups=groups))

split = np.full(len(df_clean), "train", dtype=object)
split[test_idx] = "test"

# train → train + val (75/25 of 80% = 60/20)
df_trainval = df_clean.iloc[trainval_idx].copy().reset_index(drop=True)
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx2, val_idx = next(
    gss_val.split(df_trainval, df_trainval["y_hard"],
                  groups=df_trainval[TEXT_COLUMN])
)
split[trainval_idx[val_idx]] = "val"
df_clean["split"] = split
# Stable identifier for pairing predictions across notebooks. Cleaned text is
# not unique (10,550 test rows share 10,194 distinct cleaned statements), so a
# merge on text alone is unsafe.
df_clean["id"] = np.arange(len(df_clean), dtype=np.int64)

print("\nSplit counts:")
print(df_clean["split"].value_counts())

# Leakage check
group_split_nunique = df_clean.groupby(TEXT_COLUMN)["split"].nunique()
n_leaky = (group_split_nunique > 1).sum()
assert n_leaky == 0, f"{n_leaky} text groups leak across splits!"
print("Leakage check passed.")

# Save
split_path = OUTPUT_PATH / "master_split_soft_all.csv"
df_clean.to_csv(split_path, index=False)
print(f"Saved master split → {split_path}")

# 3. ALBERT & BioBERT Models (soft training/validation/testing)


## 3.1 Shared helper functions

In [ ]:
def bootstrap_ci_mean(values: np.ndarray, n_iterations: int = 1000) -> Tuple[float, float, float]:
    """Bootstrap CI for the MEAN of a per-sample metric array."""
    values = np.asarray(values, dtype=np.float64)
    n = len(values)
    if n == 0:
        return np.nan, np.nan, np.nan
    idx_all = np.arange(n)
    boot_means = []
    for _ in range(n_iterations):
        idx = resample(idx_all)
        boot_means.append(values[idx].mean())
    boot_means = np.asarray(boot_means, dtype=np.float64)
    return float(values.mean()), float(np.percentile(boot_means, 2.5)), float(np.percentile(boot_means, 97.5))


def bootstrap_f1_ci(y_true, y_pred, n_iterations=1000, average="weighted"):
    """Return the OBSERVED F1 on the full test set plus a bootstrap 95% CI.

    The first element is the direct point estimate, not the mean of the
    bootstrap replicates; replicates are used only for the interval.
    """
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    observed = f1_score(y_true, y_pred, average=average)
    scores = []
    for _ in range(n_iterations):
        idx = resample(np.arange(len(y_true)))
        try:
            scores.append(f1_score(y_true[idx], y_pred[idx], average=average))
        except ValueError:
            continue
    if not scores:
        return observed, np.nan, np.nan
    return observed, np.percentile(scores, 2.5), np.percentile(scores, 97.5)


def bootstrap_auc_ci(y_true, y_scores, n_iterations=1000, average="macro"):
    """Observed macro AUC plus a bootstrap 95% CI (point estimate is direct)."""
    y_true, y_scores = np.asarray(y_true), np.asarray(y_scores)
    try:
        observed = roc_auc_score(y_true, y_scores, average=average, multi_class="ovr")
    except Exception:
        observed = np.nan
    scores = []
    for _ in range(n_iterations):
        idx = resample(np.arange(len(y_true)))
        try:
            scores.append(
                roc_auc_score(y_true[idx], y_scores[idx], average=average, multi_class="ovr")
            )
        except ValueError:
            continue
    if not scores:
        return observed, np.nan, np.nan
    return observed, np.percentile(scores, 2.5), np.percentile(scores, 97.5)
### 2.4.2 Plotting
def plot_confusion_matrix(y_true, y_pred, labels, title="Model", save_dir=None):
    """Works for both string labels (ML) and int labels (DL)."""
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix — {title}")
    plt.tight_layout()
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        path = os.path.join(save_dir, f"{title}_confusion_matrix.png")
        plt.savefig(path, dpi=300)
        print(f"Saved → {path}")
    plt.show()

# ── Soft metrics ─────────────────────────────────────────

def per_sample_soft_ce(probs_np, soft_targets_np, eps=1e-12):
    """Per-sample Soft CE: -sum_k soft_k * log(p_k)"""
    probs = np.clip(np.asarray(probs_np, dtype=np.float64), eps, 1.0)
    soft = np.asarray(soft_targets_np, dtype=np.float64)
    return -np.sum(soft * np.log(probs), axis=1)


def per_sample_brier(probs_np, soft_targets_np):
    """Per-sample multiclass Brier: sum_k (p_k - soft_k)^2"""
    probs = np.asarray(probs_np, dtype=np.float64)
    soft = np.asarray(soft_targets_np, dtype=np.float64)
    return np.sum((probs - soft) ** 2, axis=1)


def bootstrap_soft_ce_ci(probs_np, soft_targets_np, n_iterations=1000):
    values = per_sample_soft_ce(probs_np, soft_targets_np)
    return bootstrap_ci_mean(values, n_iterations=n_iterations)


def bootstrap_brier_ci(probs_np, soft_targets_np, n_iterations=1000):
    values = per_sample_brier(probs_np, soft_targets_np)
    return bootstrap_ci_mean(values, n_iterations=n_iterations)


# ── Plots ────────────────────────────────────────────────

def plot_confusion_matrix(y_true, y_pred, class_names, title="Model", save_dir=None):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix — {title}")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        path = os.path.join(save_dir, f"{title}_confusion_matrix.png")
        plt.savefig(path, dpi=300)
        print(f"Saved → {path}")
    plt.show()


def plot_multiclass_roc(y_true, y_score, class_names, title="Model", save_dir=None):
    y_true_bin = label_binarize(np.asarray(y_true, dtype=int),
                                classes=np.arange(len(class_names)))
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(len(class_names)):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    colors = sns.color_palette("mako", len(class_names))
    plt.figure(figsize=(10, 8))
    for i, (name, color) in enumerate(zip(class_names, colors)):
        plt.plot(fpr[i], tpr[i], color=color, lw=2,
                 label=f"{name} (AUC={roc_auc[i]:.2f})")
    plt.plot([0, 1], [0, 1], "k--", lw=2)
    plt.xlim([0, 1])
    plt.ylim([0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{title} — Multi-class ROC")
    plt.legend(loc="lower right")
    plt.tight_layout()
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        path = os.path.join(save_dir, f"{title}_roc_curve.png")
        plt.savefig(path, dpi=300)
        print(f"Saved → {path}")
    plt.show()


# ── High-level evaluators ────────────────────────────────

def evaluate_hard(y_true_hard, logits_np, class_names, model_tag="MODEL",
                  save_dir=None, n_boot=1000):
    """Hard evaluation: classification report + F1 CI + AUC CI + plots."""
    y_true_hard = np.asarray(y_true_hard, dtype=np.int64)
    probs = torch.softmax(torch.tensor(logits_np, dtype=torch.float32), dim=1).numpy()
    preds = probs.argmax(axis=1)

    print(f"\n{'='*60}")
    print(f"Evaluation (HARD): {model_tag}")
    print(f"{'='*60}")
    print(classification_report(
        y_true_hard, preds,
        labels=list(range(len(class_names))),
        target_names=class_names, digits=4, zero_division=0,
    ))

    f1_m, f1_lo, f1_hi = bootstrap_f1_ci(y_true_hard, preds, n_iterations=n_boot)
    f1_mac, f1_mac_lo, f1_mac_hi = bootstrap_f1_ci(
        y_true_hard, preds, n_iterations=n_boot, average="macro")
    auc_m, auc_lo, auc_hi = bootstrap_auc_ci(y_true_hard, probs, n_iterations=n_boot)

    print(f"Weighted F1: {f1_m:.4f}  95% CI [{f1_lo:.4f}, {f1_hi:.4f}]")
    print(f"Macro F1:    {f1_mac:.4f}  95% CI [{f1_mac_lo:.4f}, {f1_mac_hi:.4f}]")
    print(f"Macro AUC:   {auc_m:.4f}  95% CI [{auc_lo:.4f}, {auc_hi:.4f}]")

    plot_confusion_matrix(y_true_hard, preds, class_names,
                          title=f"{model_tag}_SoftAll", save_dir=save_dir)
    plot_multiclass_roc(y_true_hard, probs, class_names,
                        title=f"{model_tag}_SoftAll", save_dir=save_dir)

    return {
        "y_true": y_true_hard, "y_pred": preds, "y_prob": probs,
        "f1_weighted": f1_m, "f1_ci": (f1_lo, f1_hi),
        "f1_macro": f1_mac, "f1_macro_ci": (f1_mac_lo, f1_mac_hi),
        "auc_macro": auc_m, "auc_ci": (auc_lo, auc_hi),
    }


def evaluate_soft(probs_np, y_true_soft, model_tag="MODEL", n_boot=1000):
    """Soft evaluation: CE + Brier with bootstrap CIs."""
    probs_np = np.asarray(probs_np, dtype=np.float64)
    y_true_soft = np.asarray(y_true_soft, dtype=np.float64)

    ce_m, ce_lo, ce_hi = bootstrap_soft_ce_ci(probs_np, y_true_soft, n_iterations=n_boot)
    br_m, br_lo, br_hi = bootstrap_brier_ci(probs_np, y_true_soft, n_iterations=n_boot)

    print(f"\n--- Evaluation (SOFT): {model_tag} ---")
    print(f"Soft CE: {ce_m:.6f}  95% CI [{ce_lo:.6f}, {ce_hi:.6f}]")
    print(f"Brier:   {br_m:.6f}  95% CI [{br_lo:.6f}, {br_hi:.6f}]")

    return {
        "soft_ce": ce_m, "soft_ce_ci": (ce_lo, ce_hi),
        "brier": br_m, "brier_ci": (br_lo, br_hi),
    }

## 3.2 Data splits & labels

In [ ]:
def _extract_soft(df):
    s = df[SOFT_COLS].to_numpy(dtype=np.float32)
    return s / np.clip(s.sum(axis=1, keepdims=True), 1e-8, None)

df_train = df_clean[df_clean["split"] == "train"].copy()
df_val   = df_clean[df_clean["split"] == "val"].copy()
df_test  = df_clean[df_clean["split"] == "test"].copy()

train_texts = df_train[TEXT_COLUMN].astype(str).tolist()
val_texts   = df_val[TEXT_COLUMN].astype(str).tolist()
test_texts  = df_test[TEXT_COLUMN].astype(str).tolist()

# Soft labels for all splits
y_train_soft = _extract_soft(df_train)
y_val_soft   = _extract_soft(df_val)
y_test_soft  = _extract_soft(df_test)

# Hard labels (argmax) for evaluation
y_train_hard = df_train["y_hard"].to_numpy(dtype=np.int64)
y_val_hard   = df_val["y_hard"].to_numpy(dtype=np.int64)
y_test_hard  = df_test["y_hard"].to_numpy(dtype=np.int64)

print(f"Train soft: {y_train_soft.shape} | Val soft: {y_val_soft.shape} | Test soft: {y_test_soft.shape}")
print(f"Val  hard dist: {np.bincount(y_val_hard, minlength=NUM_CLASSES)}")
print(f"Test hard dist: {np.bincount(y_test_hard, minlength=NUM_CLASSES)}")
# ── Class weights: balanced from the hard TRAIN labels, exactly as R0 hard ──
_counts = np.bincount(y_train_hard, minlength=NUM_CLASSES).astype(np.float64)
assert (_counts > 0).all(), f"empty class in train: {_counts.tolist()}"
CLASS_WEIGHTS_DL = torch.tensor(
    len(y_train_hard) / (NUM_CLASSES * _counts), dtype=torch.float32
)
# Unweighted soft cross-entropy. Class weighting is not target-neutral for a
# distributed target: it amplifies probability mass assigned to rare classes
# and changes the target geometry. Scheduler, gradient clipping, epoch range,
# and optimizer settings remain matched to the hard-label arm.
SOFT_CLASS_WEIGHTS = None
print("Class weights:", "none" if SOFT_CLASS_WEIGHTS is None
      else [round(float(w), 3) for w in CLASS_WEIGHTS_DL])


## 3.3 Generic soft-label model componets

In [ ]:
# ── Pre-tokenization ─────────────────────────────────────
def batch_tokenize(texts, tokenizer, max_len):
    enc = tokenizer(
        [str(t) for t in texts],
        truncation=True, padding="max_length",
        max_length=max_len, return_tensors="pt",
    )
    return enc["input_ids"], enc["attention_mask"]


# ── Dataset ──────────────────────────────────────────────
class SoftLabelDataset(Dataset):
    """Pre-tokenized dataset with optional soft and hard labels."""

    def __init__(self, input_ids, attention_masks,
                 hard_labels=None, soft_labels=None):
        self.input_ids = input_ids
        self.attention_masks = attention_masks

        self.soft_labels = None
        if soft_labels is not None:
            sl = np.asarray(soft_labels, dtype=np.float32)
            rs = sl.sum(axis=1, keepdims=True)
            self.soft_labels = sl / np.clip(rs, 1e-8, None)

        self.hard_labels = None
        if hard_labels is not None:
            self.hard_labels = np.asarray(hard_labels, dtype=np.int64)
        elif self.soft_labels is not None:
            self.hard_labels = self.soft_labels.argmax(axis=1).astype(np.int64)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        item = {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_masks[idx],
        }
        if self.hard_labels is not None:
            item["labels"] = torch.tensor(self.hard_labels[idx], dtype=torch.long)
        if self.soft_labels is not None:
            item["soft_labels"] = torch.tensor(self.soft_labels[idx], dtype=torch.float32)
        return item


# ── Classifier ───────────────────────────────────────────
class TransformerClassifier(nn.Module):
    """Wrapper over the HuggingFace sequence-classification head used by R0 hard.

    Keeps the (input_ids, attention_mask) -> logits call signature so the rest of
    this notebook is unchanged, while the head itself now matches the R0 hard arm
    (AlbertForSequenceClassification / AutoModelForSequenceClassification).
    """

    def __init__(self, hf_model):
        super().__init__()
        self.model = hf_model

    def forward(self, input_ids, attention_mask):
        out = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return out.logits if hasattr(out, "logits") else out


# ── Loss function ────────────────────────────────────────
def soft_cross_entropy(logits, soft_targets, class_weights=None):
    """Soft-label CE: -sum_k w_k * soft_k * log_softmax(logits)_k

    class_weights=None reproduces the original R0 soft objective.
    With class_weights set, the reduction is the weighted mean, matching what
    PyTorch's CrossEntropyLoss(weight=...) computes in the R0 hard arm.
    """
    log_probs = torch.log_softmax(logits, dim=1)
    if class_weights is None:
        return -(soft_targets * log_probs).sum(dim=1).mean()
    w = class_weights.to(logits.device).float().unsqueeze(0)
    weighted = soft_targets * w
    num = -(weighted * log_probs).sum()
    den = weighted.sum().clamp_min(torch.finfo(torch.float32).eps)
    return num / den


# ── Inference helper ─────────────────────────────────────
def predict_logits(model, loader, device=DEVICE):
    """Run inference, return (logits, hard_labels, soft_labels) as numpy."""
    model.eval()
    all_logits, all_labels, all_soft = [], [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device, non_blocking=True)
            mask = batch["attention_mask"].to(device, non_blocking=True)

            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
                logits = model(ids, mask)

            all_logits.append(logits.float().cpu().numpy())
            if "labels" in batch:
                all_labels.append(batch["labels"].numpy())
            if "soft_labels" in batch:
                all_soft.append(batch["soft_labels"].numpy())
    return (
        np.concatenate(all_logits),
        np.concatenate(all_labels) if all_labels else None,
        np.concatenate(all_soft) if all_soft else None,
    )


# ── Training function ────────────────────────────────────
def train_soft_model(model, train_loader, val_loader,
                     lr=2e-5, max_epochs=6, patience=PATIENCE,
                     class_weights=None):
    """Soft-CE training under the R0 HARD optimizer settings.

    FP16 + GradScaler, OneCycleLR, gradient clipping at MAX_GRAD_NORM,
    best epoch by validation weighted F1 with patience-based early stopping.
    Only the target and its loss differ from the R0 hard arm.
    """
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    total_steps = max_epochs * len(train_loader)
    scheduler = (OneCycleLR(optimizer, max_lr=lr, total_steps=total_steps,
                            pct_start=ONECYCLE_PCT_START)
                 if USE_ONECYCLE else None)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)

    best_f1, best_state, best_epoch = -1.0, None, -1
    best_val_soft_loss = np.inf
    no_improve = 0

    history = {"train_soft_loss": [], "val_soft_loss": [], "val_f1w": []}

    for epoch in range(1, max_epochs + 1):
        model.train()
        losses = []
        for batch in train_loader:
            ids  = batch["input_ids"].to(DEVICE, non_blocking=True)
            mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
            soft = batch["soft_labels"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
                logits = model(ids, mask)
                loss = soft_cross_entropy(logits, soft, class_weights)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            if scheduler is not None:
                scheduler.step()
            losses.append(loss.item())

        train_loss = float(np.mean(losses))

        # Validate
        model.eval()
        val_preds, val_true, val_losses_ep = [], [], []
        with torch.no_grad():
            for batch in val_loader:
                ids  = batch["input_ids"].to(DEVICE, non_blocking=True)
                mask = batch["attention_mask"].to(DEVICE, non_blocking=True)

                with torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
                    logits = model(ids, mask)

                val_preds.extend(logits.float().argmax(dim=1).cpu().numpy())
                val_true.extend(batch["labels"].numpy())

                # Soft loss for monitoring
                if "soft_labels" in batch:
                    soft = batch["soft_labels"].to(DEVICE, non_blocking=True)
                    with torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
                        vloss = soft_cross_entropy(logits, soft, class_weights)
                    val_losses_ep.append(vloss.item())

        val_f1 = f1_score(val_true, val_preds, average="weighted")
        val_soft_loss = float(np.mean(val_losses_ep)) if val_losses_ep else np.nan

        history["train_soft_loss"].append(train_loss)
        history["val_soft_loss"].append(val_soft_loss)
        history["val_f1w"].append(val_f1)

        print(f"  Epoch {epoch}/{max_epochs} | TrainLoss={train_loss:.4f} | "
              f"ValSoftLoss={val_soft_loss:.4f} | ValF1={val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            best_val_soft_loss = val_soft_loss
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stopping at epoch {epoch}.")
                break

    if best_state:
        model.load_state_dict(best_state)
    return best_f1, model, best_epoch, history


def build_loaders(tokenizer, model_label, include_val_soft=True):
    """Tokenize and build train/val/test loaders."""
    print(f"\nTokenizing for {model_label}...")
    train_ids, train_masks = batch_tokenize(train_texts, tokenizer, MAX_TOKEN_LENGTH)
    val_ids, val_masks     = batch_tokenize(val_texts, tokenizer, MAX_TOKEN_LENGTH)
    test_ids, test_masks   = batch_tokenize(test_texts, tokenizer, MAX_TOKEN_LENGTH)

    loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                     pin_memory=True, persistent_workers=True, prefetch_factor=4)

    # Train: soft labels for training
    train_loader = DataLoader(
        SoftLabelDataset(train_ids, train_masks, soft_labels=y_train_soft),
        shuffle=True, **loader_kw)

    # Val: both soft (for soft loss monitoring) and hard (for F1)
    val_loader = DataLoader(
        SoftLabelDataset(val_ids, val_masks,
                         soft_labels=y_val_soft if include_val_soft else None,
                         hard_labels=y_val_hard),
        shuffle=False, **loader_kw)

    # Test: both soft and hard for full evaluation
    test_loader = DataLoader(
        SoftLabelDataset(test_ids, test_masks,
                         soft_labels=y_test_soft,
                         hard_labels=y_test_hard),
        shuffle=False, **loader_kw)

    return train_loader, val_loader, test_loader

def dropout_report(module):
    """Map every instantiated nn.Dropout to its probability.

    Reporting the full named map avoids guessing which layer matters: ALBERT's
    backbone dropout is 0.0 while its classifier dropout carries the sampled
    value, and the aspect head's dropout is separate from the encoder's.
    """
    report = {}
    for name, m in module.named_modules():
        if isinstance(m, nn.Dropout):
            report[name or "<root>"] = float(m.p)
    return report


def sampled_dropout_effective(module):
    """The dropout the sampled hyperparameter is meant to control.

    In AlbertForSequenceClassification the classifier dropout module is the
    top-level ``dropout`` (the backbone's own dropout is 0.0), so prefer the
    shallowest module whose name ends in "dropout"; fall back to the deepest
    encoder dropout only if no top-level one exists.
    """
    report = dropout_report(module)
    if not report:
        return None
    named = sorted(report.items(), key=lambda kv: (kv[0].count("."), len(kv[0])))
    for name, p in named:
        if "classifier" in name.lower():
            return p
    for name, p in named:
        if name.split(".")[-1].lower() == "dropout":
            return p
    return named[0][1]


## 3.4 ALBERT

In [ ]:
print("\n" + "=" * 60)
print("ALBERT — 7-Class Soft-Label Training (Mental Health)")
print("=" * 60)

### 3.4.1 Pre-tokenize

In [ ]:
albert_tokenizer = AutoTokenizer.from_pretrained(ALBERT_MODEL_NAME)
train_loader_albert, val_loader_albert, test_loader_albert = \
    build_loaders(albert_tokenizer, "ALBERT")

### 3.4.2 Hyperparameter tuning

In [ ]:
if RETRAIN:
    set_seed(RUN_SEED)
    start = time.time()

    best_f1_albert, best_params_albert, best_albert = -1.0, None, None
    best_history_albert = None

    for i in tqdm(range(N_ITERATIONS), desc="ALBERT SoftAll random search"):
        hp = {
            "lr": float(10 ** np.random.uniform(*TRANSFORMER_PARAM_SPACE["lr"])),
            "epochs": int(np.random.randint(*TRANSFORMER_PARAM_SPACE["epochs"])),
            "dropout": float(np.random.uniform(*TRANSFORMER_PARAM_SPACE["dropout"])),
        }
        print(f"ALBERT trial {i + 1}/{N_ITERATIONS}: {hp}")

        hf = AlbertForSequenceClassification.from_pretrained(
            ALBERT_MODEL_NAME, num_labels=NUM_CLASSES,
            classifier_dropout_prob=hp["dropout"],
        )
        hp["dropout_modules"] = dropout_report(hf)
        if USE_COMPILE:
            hf = torch.compile(hf, mode="reduce-overhead")
        model = TransformerClassifier(hf)

        f1_val, model, best_ep, history = train_soft_model(
            model, train_loader_albert, val_loader_albert,
            lr=hp["lr"], max_epochs=hp["epochs"],
            class_weights=SOFT_CLASS_WEIGHTS,
        )

        if f1_val > best_f1_albert:
            best_f1_albert = f1_val
            best_params_albert = {"seed": RUN_SEED, "trial": i + 1, "best_epoch": best_ep,
                                  "class_weighting": "none_unweighted_soft_ce", **hp}
            best_albert = model
            best_history_albert = history
            torch.save(model.state_dict(), MODEL_PATHS["albert"]["model"])
            with open(MODEL_PATHS["albert"]["params"], "w") as _pf:
                json.dump(best_params_albert, _pf, indent=2)
            pd.DataFrame(history).to_csv(OUTPUT_PATH / "history_albert.csv", index=False)
            print(f"  New best ALBERT: ValF1={f1_val:.4f}")

        gc.collect()
        torch.cuda.empty_cache()

    print(f"ALBERT random search: {time.time()-start:.1f}s | Best F1={best_f1_albert:.4f}")
    print(f"Best params: {best_params_albert}")

    with open(MODEL_PATHS["albert"]["params"], "w") as f:
        json.dump(best_params_albert, f, indent=2)

else:
    with open(MODEL_PATHS["albert"]["params"]) as f:
        best_params_albert = json.load(f)
    hf = AlbertForSequenceClassification.from_pretrained(
        ALBERT_MODEL_NAME, num_labels=NUM_CLASSES,
        classifier_dropout_prob=best_params_albert["dropout"],
    )
    if USE_COMPILE:
        hf = torch.compile(hf, mode="reduce-overhead")
    best_albert = TransformerClassifier(hf)
    best_albert.load_state_dict(torch.load(MODEL_PATHS["albert"]["model"], map_location=DEVICE))

best_albert.to(DEVICE)
best_albert.eval()


### 3.4.3 Model evaluation

In [ ]:
# Hard evaluation
test_logits_a, test_y_hard_a, test_y_soft_a = predict_logits(
    best_albert, test_loader_albert, DEVICE)
test_probs_a = torch.softmax(torch.tensor(test_logits_a), dim=1).numpy()

albert_hard_results = evaluate_hard(
    test_y_hard_a, test_logits_a, CLASS_NAMES,
    model_tag="ALBERT_TEST", save_dir=str(OUTPUT_PATH))

# Soft evaluation
albert_soft_results = evaluate_soft(
    test_probs_a, test_y_soft_a, model_tag="ALBERT_TEST")

# Val evaluation (for comparison)
val_logits_a, val_y_hard_a, val_y_soft_a = predict_logits(
    best_albert, val_loader_albert, DEVICE)
val_probs_a = torch.softmax(torch.tensor(val_logits_a), dim=1).numpy()

albert_val_hard = evaluate_hard(
    val_y_hard_a, val_logits_a, CLASS_NAMES,
    model_tag="ALBERT_VAL", save_dir=str(OUTPUT_PATH))
albert_val_soft = evaluate_soft(
    val_probs_a, val_y_soft_a, model_tag="ALBERT_VAL")

## 3.5 BioBERT

In [ ]:
print("\n" + "=" * 60)
print("BioBERT — 7-Class Soft-Label Training (Mental Health)")
print("=" * 60)

### 3.5.1 Pre-tokenize

In [ ]:
biobert_tokenizer = AutoTokenizer.from_pretrained(BIOBERT_MODEL_NAME)
train_loader_biobert, val_loader_biobert, test_loader_biobert = \
    build_loaders(biobert_tokenizer, "BioBERT")

### 3.5.2 Hyperparameter tuning

In [ ]:
if RETRAIN:
    set_seed(RUN_SEED)
    start = time.time()

    best_f1_biobert, best_params_biobert, best_biobert = -1.0, None, None
    best_history_biobert = None

    for i in tqdm(range(N_ITERATIONS), desc="BioBERT SoftAll random search"):
        hp = {
            "lr": float(10 ** np.random.uniform(*TRANSFORMER_PARAM_SPACE["lr"])),
            "epochs": int(np.random.randint(*TRANSFORMER_PARAM_SPACE["epochs"])),
            "dropout": float(np.random.uniform(*TRANSFORMER_PARAM_SPACE["dropout"])),
        }
        print(f"BioBERT trial {i + 1}/{N_ITERATIONS}: {hp}")

        biobert_config = AutoConfig.from_pretrained(
            BIOBERT_MODEL_NAME, num_labels=NUM_CLASSES,
        )
        # Dropout must be set on the config BEFORE instantiation; assigning it to
        # model.config afterwards leaves the constructed nn.Dropout modules at the
        # pretrained default (the R0 behaviour, corrected here in both arms).
        biobert_config.hidden_dropout_prob = hp["dropout"]
        biobert_config.attention_probs_dropout_prob = hp["dropout"]

        hf = AutoModelForSequenceClassification.from_pretrained(
            BIOBERT_MODEL_NAME, config=biobert_config,
        )
        hp["dropout_modules"] = dropout_report(hf)
        if USE_COMPILE:
            hf = torch.compile(hf, mode="reduce-overhead")
        model = TransformerClassifier(hf)

        f1_val, model, best_ep, history = train_soft_model(
            model, train_loader_biobert, val_loader_biobert,
            lr=hp["lr"], max_epochs=hp["epochs"],
            class_weights=SOFT_CLASS_WEIGHTS,
        )

        if f1_val > best_f1_biobert:
            best_f1_biobert = f1_val
            best_params_biobert = {"seed": RUN_SEED, "trial": i + 1, "best_epoch": best_ep,
                                   "class_weighting": "none_unweighted_soft_ce", **hp}
            best_biobert = model
            best_history_biobert = history
            torch.save(model.state_dict(), MODEL_PATHS["biobert"]["model"])
            with open(MODEL_PATHS["biobert"]["params"], "w") as _pf:
                json.dump(best_params_biobert, _pf, indent=2)
            pd.DataFrame(history).to_csv(OUTPUT_PATH / "history_biobert.csv", index=False)
            print(f"  New best BioBERT: ValF1={f1_val:.4f}")

        gc.collect()
        torch.cuda.empty_cache()

    print(f"BioBERT random search: {time.time()-start:.1f}s | Best F1={best_f1_biobert:.4f}")
    print(f"Best params: {best_params_biobert}")

    with open(MODEL_PATHS["biobert"]["params"], "w") as f:
        json.dump(best_params_biobert, f, indent=2)

else:
    with open(MODEL_PATHS["biobert"]["params"]) as f:
        best_params_biobert = json.load(f)
    biobert_config = AutoConfig.from_pretrained(
        BIOBERT_MODEL_NAME, num_labels=NUM_CLASSES,
    )
    biobert_config.hidden_dropout_prob = best_params_biobert["dropout"]
    biobert_config.attention_probs_dropout_prob = best_params_biobert["dropout"]
    hf = AutoModelForSequenceClassification.from_pretrained(
        BIOBERT_MODEL_NAME, config=biobert_config,
    )
    if USE_COMPILE:
        hf = torch.compile(hf, mode="reduce-overhead")
    best_biobert = TransformerClassifier(hf)
    best_biobert.load_state_dict(torch.load(MODEL_PATHS["biobert"]["model"], map_location=DEVICE))

best_biobert.to(DEVICE)
best_biobert.eval()


### 3.4.3 Model evaluation

In [ ]:
# Hard evaluation
test_logits_b, test_y_hard_b, test_y_soft_b = predict_logits(
    best_biobert, test_loader_biobert, DEVICE)
test_probs_b = torch.softmax(torch.tensor(test_logits_b), dim=1).numpy()

biobert_hard_results = evaluate_hard(
    test_y_hard_b, test_logits_b, CLASS_NAMES,
    model_tag="BioBERT_TEST", save_dir=str(OUTPUT_PATH))

# Soft evaluation
biobert_soft_results = evaluate_soft(
    test_probs_b, test_y_soft_b, model_tag="BioBERT_TEST")

# Val evaluation
val_logits_b, val_y_hard_b, val_y_soft_b = predict_logits(
    best_biobert, val_loader_biobert, DEVICE)
val_probs_b = torch.softmax(torch.tensor(val_logits_b), dim=1).numpy()

biobert_val_hard = evaluate_hard(
    val_y_hard_b, val_logits_b, CLASS_NAMES,
    model_tag="BioBERT_VAL", save_dir=str(OUTPUT_PATH))
biobert_val_soft = evaluate_soft(
    val_probs_b, val_y_soft_b, model_tag="BioBERT_VAL")

## 3.6 Save prediction

In [ ]:
df_test_out = df_test[["id", TEXT_COLUMN, "y_hard"]].copy()
df_test_out["albert_pred"] = albert_hard_results["y_pred"]
df_test_out["biobert_pred"] = biobert_hard_results["y_pred"]

for i, name in enumerate(CLASS_NAMES):
    df_test_out[f"albert_p_{name}"] = albert_hard_results["y_prob"][:, i]
    df_test_out[f"biobert_p_{name}"] = biobert_hard_results["y_prob"][:, i]

pred_path = OUTPUT_PATH / "test_predictions_soft_all.csv"
df_test_out.to_csv(pred_path, index=False)
print(f"\nSaved predictions → {pred_path}")

print("\n" + "=" * 60)
print("SUMMARY — 7-Class Soft-Label All (Mental Health)")
print("=" * 60)

for model_name, hard_r, soft_r in [
    ("ALBERT", albert_hard_results, albert_soft_results),
    ("BioBERT", biobert_hard_results, biobert_soft_results),
]:
    print(f"\n{model_name}:")
    print(f"  Weighted F1: {hard_r['f1_weighted']:.4f} "
          f"[{hard_r['f1_ci'][0]:.4f}, {hard_r['f1_ci'][1]:.4f}]")
    if "f1_macro" in hard_r:
        print(f"  Macro F1:    {hard_r['f1_macro']:.4f} "
                f"[{hard_r['f1_macro_ci'][0]:.4f}, {hard_r['f1_macro_ci'][1]:.4f}]")
    print(f"  Macro AUC:   {hard_r['auc_macro']:.4f} "
          f"[{hard_r['auc_ci'][0]:.4f}, {hard_r['auc_ci'][1]:.4f}]")
    print(f"  Soft CE:     {soft_r['soft_ce']:.6f} "
          f"[{soft_r['soft_ce_ci'][0]:.6f}, {soft_r['soft_ce_ci'][1]:.6f}]")
    print(f"  Brier:       {soft_r['brier']:.6f} "
          f"[{soft_r['brier_ci'][0]:.6f}, {soft_r['brier_ci'][1]:.6f}]")